# Notebook 01: Data Pipeline Design

**Module:** 11 Production ML  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Design reproducible ML data pipelines
2. Understand ETL vs ELT for GeoSpatial data
3. Map water-bodies tile_and_mask.py workflow
4. Identify pipeline failure points

## Production Data Pipeline

```
Raw data → Validate → Transform → Version → Train/Val split → Loader
```

**Principles:**
- **Idempotent:** Re-run produces same output given same inputs
- **Validated:** Schema checks at every stage
- **Lineage:** Track which raw data produced which tiles
- **Reproducible:** Fixed seeds, frozen configs

In [ ]:
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## water-bodies-detection Pipeline

```
Planet GeoTIFF + shapefile
  → tile_and_mask.py (GDAL/rasterio)
     • Band selection [2,3,4,6,7,8]
     • 512×512 tiles, 50% overlap
     • Dual masks: aqua + boundary
     • Negative tile sampling (20%)
  → tiles/, masks_aqua/, masks_boundary/
  → dataset.py (Albumentations)
  → train.py
```

See: `water-bodies-detection/tile_and_mask.py`

In [ ]:
# Pipeline stage checklist
def validate_pipeline_stages(stages):
    for name, checks in stages.items():
        print(f'{name}: {sum(checks)}/{len(checks)} checks passed')

stages = {
    'ingest': [True, True, False],  # example
    'tile': [True, True, True],
    'train': [True, False, True],
}
validate_pipeline_stages(stages)

## GeoSpatial-Specific Concerns

- CRS consistency (EPSG codes)
- NoData handling in multispectral bands
- Alpha band masking (cloud/invalid pixels)
- meters_per_pixel for boundary width
- Large raster memory — tile before load

---

## Summary

Production data pipelines are idempotent, validated, and traceable.

**Next:** [02_Training_Pipeline_Architecture.ipynb](02_Training_Pipeline_Architecture.ipynb)